# Chapter 10 — Reduction

Based on chapter 10 of *Programming Massively Parallel Processors* (Hwu, Kirk, El Hajj). Every program in this folder is a single `.cu` file that runs on an NVIDIA GPU (Google Colab) **and** on any laptop without a GPU, through the CPU emulation in [`../common/cuda_compat.h`](../common/cuda_compat.h).

This notebook goes **step by step**: 12 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/08_reduction/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 12: `01_sequential_reduction.cu`

`sum_sequential` and the general form `reduce_sequential` with an operator and its identity (0 for the sum, minus infinity for the maximum) on 16 values; prints the accumulator after every element. (slide: *What you already know: operator and identity*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_sequential_reduction.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cmath>
#include <cstdio>
#include <numeric>

constexpr unsigned int N = 16;
const float INPUT[N] = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 8, 1, 9, 2, 4, 6};

constexpr float SUM_IDENTITY = 0.0f;
const float MAX_IDENTITY = -INFINITY;

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

float add(float a, float b) {
  return a + b;
}

float maximum(float a, float b) {
  return std::fmax(a, b);
}

template <class Operator>
float reduce_sequential(const float* input, unsigned int n, float identity, Operator op) {
  float acc = identity;
  for (unsigned int i = 0; i < n; ++i) {
    acc = op(acc, input[i]);
  }
  return acc;
}

static void printValues(const char* label, const float* values, unsigned int n) {
  std::printf("%-12s", label);
  for (unsigned int i = 0; i < n; ++i) {
    std::printf(" %3.0f", values[i]);
  }
  std::printf("\n");
}

template <class Operator>
static void printRunningValues(const char* label, float identity, Operator op) {
  float running[N];
  for (unsigned int n = 1; n <= N; ++n) {
    running[n - 1] = reduce_sequential(INPUT, n, identity, op);
  }
  printValues(label, running, N);
}

int main() {
  printValues("input", INPUT, N);
  std::printf("\nthe accumulator after each element (one step per element, %u steps):\n", N);
  printRunningValues("sum (0, +)", SUM_IDENTITY, add);
  printRunningValues("max (-inf)", MAX_IDENTITY, maximum);

  const float sum = sum_sequential(INPUT, N);
  const float generalSum = reduce_sequential(INPUT, N, SUM_IDENTITY, add);
  const float generalMax = reduce_sequential(INPUT, N, MAX_IDENTITY, maximum);
  const float expectedSum = std::accumulate(INPUT, INPUT + N, 0.0f);
  const float expectedMax = *std::max_element(INPUT, INPUT + N);

  std::printf("\nsum_sequential:              %g\n", sum);
  std::printf("reduce_sequential with +:    %g (std::accumulate: %g)\n", generalSum, expectedSum);
  std::printf("reduce_sequential with max:  %g (std::max_element: %g)\n", generalMax, expectedMax);

  const bool ok = sum == expectedSum && generalSum == expectedSum && generalMax == expectedMax;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_sequential_reduction.cu -o 01_sequential_reduction && ./01_sequential_reduction

**What to look at in the output:** The sum accumulator changes at every one of the 16 steps and ends at 62; the max accumulator changes only when a larger value appears and ends at 9.

## Step 2 of 12: `02_simple_kernel.cu`

`SimpleSumReductionKernel`: one block of 8 threads for 16 elements, thread t owns `input[2t]`, the stride doubles 1, 2, 4, 8. `SimpleSumReductionKernelPrinting` prints the array after every stride; its second barrier is there only for the printing. (slide: *Code: the simple kernel*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_simple_kernel.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int N = 16;
constexpr unsigned int THREADS = N / 2;
const float INPUT[N] = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 8, 1, 9, 2, 4, 6};

__global__ void SimpleSumReductionKernel(float* input, float* output) {
  unsigned int i = 2 * threadIdx.x;
  for (unsigned int stride = 1; stride <= blockDim.x; stride *= 2) {
    if (threadIdx.x % stride == 0) {
      input[i] += input[i + stride];
    }
    __syncthreads();
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

__device__ void printArray(const char* label,
                           unsigned int stride,
                           const float* values,
                           unsigned int n) {
  printf("%s %2u:", label, stride);
  for (unsigned int k = 0; k < n; ++k) {
    printf(" %3.0f", values[k]);
  }
  printf("\n");
}

__global__ void SimpleSumReductionKernelPrinting(float* input, float* output) {
  unsigned int i = 2 * threadIdx.x;
  for (unsigned int stride = 1; stride <= blockDim.x; stride *= 2) {
    if (threadIdx.x % stride == 0) {
      input[i] += input[i + stride];
    }
    __syncthreads();

    if (threadIdx.x == 0) {
      printArray("after stride", stride, input, 2 * blockDim.x);
    }
    __syncthreads();
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  CHECK(cudaMemcpy(input_d, INPUT, N * sizeof(float), cudaMemcpyHostToDevice));
  launch(kernel, dim3(1), dim3(THREADS), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  std::printf("one block of %u threads, %u elements; thread t owns input[2t]\n", THREADS, N);
  std::printf("%-15s", "input");
  for (unsigned int k = 0; k < N; ++k) {
    std::printf(" %3.0f", INPUT[k]);
  }
  std::printf("\n");
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, N * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));

  const float printedSum = runKernel(SimpleSumReductionKernelPrinting, input_d, output_d);
  const float bookSum = runKernel(SimpleSumReductionKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(INPUT, N);
  std::printf("SimpleSumReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_simple_kernel.cu -o 02_simple_kernel && ./02_simple_kernel

**What to look at in the output:** After stride 1 the pair sums sit at the even positions, after stride 2 at the multiples of 4, and so on; the total 62 ends in `input[0]` and the odd positions never change.

## Step 3 of 12: `03_convergent_kernel.cu`

`ConvergentSumReductionKernel`: thread t owns `input[t]` and the stride halves 8, 4, 2, 1, so the active threads stay consecutive. Printed after every stride, as in step 2. (slide: *Code: the convergent kernel*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_convergent_kernel.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int N = 16;
constexpr unsigned int THREADS = N / 2;
const float INPUT[N] = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 8, 1, 9, 2, 4, 6};

__global__ void ConvergentSumReductionKernel(float* input, float* output) {
  unsigned int i = threadIdx.x;
  for (unsigned int stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x < stride) {
      input[i] += input[i + stride];
    }
    __syncthreads();
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

__device__ void printArray(const char* label,
                           unsigned int stride,
                           const float* values,
                           unsigned int n) {
  printf("%s %2u:", label, stride);
  for (unsigned int k = 0; k < n; ++k) {
    printf(" %3.0f", values[k]);
  }
  printf("\n");
}

__global__ void ConvergentSumReductionKernelPrinting(float* input, float* output) {
  unsigned int i = threadIdx.x;
  for (unsigned int stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x < stride) {
      input[i] += input[i + stride];
    }
    __syncthreads();

    if (threadIdx.x == 0) {
      printArray("after stride", stride, input, 2 * blockDim.x);
    }
    __syncthreads();
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  CHECK(cudaMemcpy(input_d, INPUT, N * sizeof(float), cudaMemcpyHostToDevice));
  launch(kernel, dim3(1), dim3(THREADS), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  std::printf("one block of %u threads, %u elements; thread t owns input[t]\n", THREADS, N);
  std::printf("%-15s", "input");
  for (unsigned int k = 0; k < N; ++k) {
    std::printf(" %3.0f", INPUT[k]);
  }
  std::printf("\n");
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, N * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));

  const float printedSum = runKernel(ConvergentSumReductionKernelPrinting, input_d, output_d);
  const float bookSum = runKernel(ConvergentSumReductionKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(INPUT, N);
  std::printf("ConvergentSumReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_convergent_kernel.cu -o 03_convergent_kernel && ./03_convergent_kernel

**What to look at in the output:** The partial sums stay packed on the left: positions 0 to 7 after stride 8, 0 to 3 after stride 4; `input[0]` ends at 62, as in step 2.

## Step 4 of 12: `04_shared_memory_kernel.cu`

`SharedMemorySumReductionKernel`: the first addition happens while loading into `input_s`, the partial sums stay in shared memory, and the input is not modified. The printing copy shows `input_s` after the load and after every stride. (slide: *Code: partial sums in shared memory*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_shared_memory_kernel.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int BLOCK_DIM = 8;
constexpr unsigned int N = 2 * BLOCK_DIM;
const float INPUT[N] = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 8, 1, 9, 2, 4, 6};

__global__ void SharedMemorySumReductionKernel(float* input, float* output) {
  __shared__ float input_s[BLOCK_DIM];
  unsigned int t = threadIdx.x;
  input_s[t] = input[t] + input[t + BLOCK_DIM];
  for (unsigned int stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (threadIdx.x < stride) {
      input_s[t] += input_s[t + stride];
    }
  }
  if (threadIdx.x == 0) {
    *output = input_s[0];
  }
}

__device__ void printValues(const float* values, unsigned int n) {
  for (unsigned int k = 0; k < n; ++k) {
    printf(" %3.0f", values[k]);
  }
  printf("\n");
}

__global__ void SharedMemorySumReductionKernelPrinting(float* input, float* output) {
  __shared__ float input_s[BLOCK_DIM];
  unsigned int t = threadIdx.x;
  input_s[t] = input[t] + input[t + BLOCK_DIM];
  __syncthreads();
  if (threadIdx.x == 0) {
    printf("input_s, loaded:  ");
    printValues(input_s, BLOCK_DIM);
  }

  for (unsigned int stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (threadIdx.x < stride) {
      input_s[t] += input_s[t + stride];
    }
    __syncthreads();

    if (threadIdx.x == 0) {
      printf("after stride %2u:  ", stride);
      printValues(input_s, BLOCK_DIM);
    }
  }
  if (threadIdx.x == 0) {
    *output = input_s[0];
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  launch(kernel, dim3(1), dim3(BLOCK_DIM), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  std::printf(
      "one block of BLOCK_DIM = %u threads, %u elements; thread t owns input_s[t]\n", BLOCK_DIM, N);
  std::printf("%-18s", "input");
  for (unsigned int k = 0; k < N; ++k) {
    std::printf(" %3.0f", INPUT[k]);
  }
  std::printf("\n");
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, N * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));
  CHECK(cudaMemcpy(input_d, INPUT, N * sizeof(float), cudaMemcpyHostToDevice));

  const float printedSum = runKernel(SharedMemorySumReductionKernelPrinting, input_d, output_d);
  const float bookSum = runKernel(SharedMemorySumReductionKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(INPUT, N);
  std::printf("SharedMemorySumReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_shared_memory_kernel.cu -o 04_shared_memory_kernel && ./04_shared_memory_kernel

**What to look at in the output:** `input_s` holds only 8 values because the load already added the two halves; the stride rows match step 3, but the input array is not modified.

## Step 5 of 12: `05_warp_reduce.cu`

`warp_reduce` with `__shfl_down_sync` on one warp of 32 values: five shuffles, no shared memory, no barrier. The printing copy shows every lane after each stride; only lane 0 ends with the total, because a lane whose partner lane i + stride is outside the warp adds its own value. (slide: *Warp-level primitives: `__shfl_down_sync`*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_warp_reduce.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int WARP_SIZE = 32;
constexpr unsigned int FULL_WARP_MASK = 0xffffffff;
const float INPUT[WARP_SIZE] = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 8, 1, 9, 2, 4, 6,
                                5, 2, 0, 7, 3, 8, 1, 4, 6, 2, 9, 3, 0, 5, 7, 1};

__device__ unsigned int laneIdx() {
  return threadIdx.x % WARP_SIZE;
}

__device__ __inline__ float warp_reduce(float val) {
  float partialSum = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialSum += __shfl_down_sync(FULL_WARP_MASK, partialSum, stride);
  }
  return partialSum;
}

__device__ void printLanes(unsigned int stride, float partialSum) {
  __shared__ float lanes_s[WARP_SIZE];
  lanes_s[laneIdx()] = partialSum;
  __syncwarp();
  if (laneIdx() == 0) {
    printf("after stride %2u:", stride);
    for (unsigned int lane = 0; lane < WARP_SIZE; ++lane) {
      printf(" %3.0f", lanes_s[lane]);
    }
    printf("\n");
  }
  __syncwarp();
}

__device__ float warp_reduce_printing(float val) {
  float partialSum = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialSum += __shfl_down_sync(FULL_WARP_MASK, partialSum, stride);
    printLanes(stride, partialSum);
  }
  return partialSum;
}

__global__ void WarpReduceKernel(float* input, float* output) {
  float partialSum = warp_reduce(input[threadIdx.x]);
  if (threadIdx.x == 0) {
    *output = partialSum;
  }
}

__global__ void WarpReduceKernelPrinting(float* input, float* output) {
  float partialSum = warp_reduce_printing(input[threadIdx.x]);
  if (threadIdx.x == 0) {
    *output = partialSum;
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  launch(kernel, dim3(1), dim3(WARP_SIZE), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  std::printf(
      "one warp of %u lanes, one value per lane; lane i adds the value of lane i + stride\n",
      WARP_SIZE);
  std::printf("%-16s", "lane values");
  for (unsigned int lane = 0; lane < WARP_SIZE; ++lane) {
    std::printf(" %3.0f", INPUT[lane]);
  }
  std::printf("\n");
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, WARP_SIZE * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));
  CHECK(cudaMemcpy(input_d, INPUT, WARP_SIZE * sizeof(float), cudaMemcpyHostToDevice));

  const float printedSum = runKernel(WarpReduceKernelPrinting, input_d, output_d);
  const float bookSum = runKernel(WarpReduceKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(INPUT, WARP_SIZE);
  std::printf("warp_reduce: %g in lane 0, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_warp_reduce.cu -o 05_warp_reduce && ./05_warp_reduce

**What to look at in the output:** After each stride only the lower lanes hold useful partial sums; lane 0 ends at 125 and the other lanes hold values that nobody uses.

## Step 6 of 12: `06_warp_level_kernel.cu`

`WarpLevelSumReductionKernel`: the shared-memory tree stops at stride 32 and warp 0 finishes with `warp_reduce`. 64 threads and 128 elements; the printing copy shows the 32 partial sums that warp 0 receives. (slide: *Code: the warp-level kernel*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 06_warp_level_kernel.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

constexpr unsigned int WARP_SIZE = 32;
constexpr unsigned int FULL_WARP_MASK = 0xffffffff;
constexpr unsigned int BLOCK_DIM = 64;
constexpr unsigned int N = 2 * BLOCK_DIM;
constexpr unsigned int VALUES_PER_ROW = 16;
constexpr unsigned int DIGITS = 10;

__device__ unsigned int warpIdx() {
  return threadIdx.x / WARP_SIZE;
}

__device__ __inline__ float warp_reduce(float val) {
  float partialSum = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialSum += __shfl_down_sync(FULL_WARP_MASK, partialSum, stride);
  }
  return partialSum;
}

__global__ void WarpLevelSumReductionKernel(float* input, float* output) {
  __shared__ float input_s[BLOCK_DIM];
  unsigned int t = threadIdx.x;
  input_s[t] = input[t] + input[t + BLOCK_DIM];
  for (unsigned int stride = blockDim.x / 2; stride >= WARP_SIZE; stride /= 2) {
    __syncthreads();
    if (threadIdx.x < stride) {
      input_s[t] += input_s[t + stride];
    }
  }
  __syncthreads();
  if (warpIdx() == 0) {
    float partialSum = input_s[t];
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      *output = partialSum;
    }
  }
}

__global__ void WarpLevelSumReductionKernelPrinting(float* input, float* output) {
  __shared__ float input_s[BLOCK_DIM];
  unsigned int t = threadIdx.x;
  input_s[t] = input[t] + input[t + BLOCK_DIM];
  for (unsigned int stride = blockDim.x / 2; stride >= WARP_SIZE; stride /= 2) {
    __syncthreads();
    if (threadIdx.x < stride) {
      input_s[t] += input_s[t + stride];
    }
  }
  __syncthreads();
  if (threadIdx.x == 0) {
    printf("input_s[0..31], the partial sums that warp 0 reduces with shuffles:\n");
    for (unsigned int lane = 0; lane < WARP_SIZE; ++lane) {
      printf(" %3.0f", input_s[lane]);
    }
    printf("\n");
  }
  if (warpIdx() == 0) {
    float partialSum = input_s[t];
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      *output = partialSum;
    }
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

static std::vector<float> makeInput() {
  std::vector<float> input(N);
  for (unsigned int i = 0; i < N; ++i) {
    input[i] = static_cast<float>((i * 7 + 3) % DIGITS);
  }
  return input;
}

static void printInput(const std::vector<float>& input) {
  std::printf("input (%u elements):\n", N);
  for (unsigned int i = 0; i < N; ++i) {
    std::printf(" %3.0f", input[i]);
    if ((i + 1) % VALUES_PER_ROW == 0) {
      std::printf("\n");
    }
  }
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  launch(kernel, dim3(1), dim3(BLOCK_DIM), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  const std::vector<float> input = makeInput();
  std::printf("one block of BLOCK_DIM = %u threads (%u warps)\n", BLOCK_DIM, BLOCK_DIM / WARP_SIZE);
  printInput(input);
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, N * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));
  CHECK(cudaMemcpy(input_d, input.data(), N * sizeof(float), cudaMemcpyHostToDevice));

  const float printedSum = runKernel(WarpLevelSumReductionKernelPrinting, input_d, output_d);
  const float bookSum = runKernel(WarpLevelSumReductionKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(input.data(), N);
  std::printf("WarpLevelSumReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 06_warp_level_kernel.cu -o 06_warp_level_kernel && ./06_warp_level_kernel

**What to look at in the output:** The 32 values of `input_s` add up to the final 570.

## Step 7 of 12: `07_two_stage_warp_kernel.cu`

`TwoStageWarpLevelSumReductionKernel`: every warp reduces its 64 elements in registers, lane 0 stores the warp sum in `partialSums_s`, one barrier, then warp 0 reduces the warp sums. 128 threads (4 warps) and 256 elements; with fewer than 1024 threads the second stage needs the guard from the slides (`t < NUM_WARPS`, else 0). (slide: *Code: two-stage warp-wide reduction*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 07_two_stage_warp_kernel.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

constexpr unsigned int WARP_SIZE = 32;
constexpr unsigned int FULL_WARP_MASK = 0xffffffff;
constexpr unsigned int BLOCK_DIM = 128;
constexpr unsigned int NUM_WARPS = BLOCK_DIM / WARP_SIZE;
constexpr unsigned int N = 2 * BLOCK_DIM;
constexpr unsigned int VALUES_PER_ROW = 16;
constexpr unsigned int DIGITS = 10;

__device__ unsigned int warpIdx() {
  return threadIdx.x / WARP_SIZE;
}

__device__ unsigned int laneIdx() {
  return threadIdx.x % WARP_SIZE;
}

__device__ __inline__ float warp_reduce(float val) {
  float partialSum = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialSum += __shfl_down_sync(FULL_WARP_MASK, partialSum, stride);
  }
  return partialSum;
}

__global__ void TwoStageWarpLevelSumReductionKernel(float* input, float* output) {
  unsigned int t = threadIdx.x;
  float partialSum = input[t] + input[t + BLOCK_DIM];
  partialSum = warp_reduce(partialSum);

  __shared__ float partialSums_s[NUM_WARPS];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    float partialSum = t < NUM_WARPS ? partialSums_s[t] : 0.0f;
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      *output = partialSum;
    }
  }
}

__global__ void TwoStageWarpLevelSumReductionKernelPrinting(float* input, float* output) {
  unsigned int t = threadIdx.x;
  float partialSum = input[t] + input[t + BLOCK_DIM];
  partialSum = warp_reduce(partialSum);

  __shared__ float partialSums_s[NUM_WARPS];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (threadIdx.x == 0) {
    printf("stage 1, partialSums_s (one sum per warp):");
    for (unsigned int warp = 0; warp < NUM_WARPS; ++warp) {
      printf(" %g", partialSums_s[warp]);
    }
    printf("\n");
  }
  if (warpIdx() == 0) {
    float partialSum = t < NUM_WARPS ? partialSums_s[t] : 0.0f;
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      printf("stage 2, warp 0 reduces them to %g\n", partialSum);
      *output = partialSum;
    }
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

static std::vector<float> makeInput() {
  std::vector<float> input(N);
  for (unsigned int i = 0; i < N; ++i) {
    input[i] = static_cast<float>((i * 7 + 3) % DIGITS);
  }
  return input;
}

static void printInput(const std::vector<float>& input) {
  std::printf("input (%u elements):\n", N);
  for (unsigned int i = 0; i < N; ++i) {
    std::printf(" %3.0f", input[i]);
    if ((i + 1) % VALUES_PER_ROW == 0) {
      std::printf("\n");
    }
  }
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  launch(kernel, dim3(1), dim3(BLOCK_DIM), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  const std::vector<float> input = makeInput();
  std::printf("one block of BLOCK_DIM = %u threads (%u warps)\n", BLOCK_DIM, NUM_WARPS);
  printInput(input);
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, N * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));
  CHECK(cudaMemcpy(input_d, input.data(), N * sizeof(float), cudaMemcpyHostToDevice));

  const float printedSum =
      runKernel(TwoStageWarpLevelSumReductionKernelPrinting, input_d, output_d);
  const float bookSum = runKernel(TwoStageWarpLevelSumReductionKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(input.data(), N);
  std::printf("TwoStageWarpLevelSumReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 07_two_stage_warp_kernel.cu -o 07_two_stage_warp_kernel && ./07_two_stage_warp_kernel

**What to look at in the output:** The four warp sums of stage 1 add up to the stage 2 result, 1148.

## Step 8 of 12: `08_multi_block_atomic.cu`

`reduce_kernel`: 4 blocks, each reduces its segment of 2 x BLOCK_DIM elements and its thread 0 adds the block sum to `*output` with a device-scope `cuda::atomic_ref` `fetch_add`. The printing copy shows each block sum and the old value `fetch_add` returned, which depends on the order in which the blocks arrive. (slide: *Code: multi-block reduction with `atomic_ref`*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 08_multi_block_atomic.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

constexpr unsigned int WARP_SIZE = 32;
constexpr unsigned int FULL_WARP_MASK = 0xffffffff;
constexpr unsigned int BLOCK_DIM = 32;
constexpr unsigned int NUM_WARPS = BLOCK_DIM / WARP_SIZE;
constexpr unsigned int NUM_BLOCKS = 4;
constexpr unsigned int INPUT_LENGTH = NUM_BLOCKS * 2 * BLOCK_DIM;
constexpr unsigned int VALUES_PER_ROW = 16;
constexpr unsigned int DIGITS = 10;

__device__ __inline__ float warp_reduce(float val) {
  float partialSum = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialSum += __shfl_down_sync(FULL_WARP_MASK, partialSum, stride);
  }
  return partialSum;
}

__global__ void reduce_kernel(float* input, float* output, unsigned int N) {
  (void)N;

  unsigned int segment = 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = input[i] + input[i + BLOCK_DIM];
  partialSum = warp_reduce(partialSum);

  __shared__ float partialSums_s[NUM_WARPS];
  if (threadIdx.x % WARP_SIZE == 0) {
    partialSums_s[threadIdx.x / WARP_SIZE] = partialSum;
  }
  __syncthreads();

  if (threadIdx.x < WARP_SIZE) {
    float partialSum = threadIdx.x < NUM_WARPS ? partialSums_s[threadIdx.x] : 0.0f;
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      output_ref.fetch_add(partialSum, cuda::memory_order_relaxed);
    }
  }
}

__global__ void reduce_kernel_printing(float* input, float* output, unsigned int N) {
  (void)N;

  unsigned int segment = 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = input[i] + input[i + BLOCK_DIM];
  partialSum = warp_reduce(partialSum);

  __shared__ float partialSums_s[NUM_WARPS];
  if (threadIdx.x % WARP_SIZE == 0) {
    partialSums_s[threadIdx.x / WARP_SIZE] = partialSum;
  }
  __syncthreads();

  if (threadIdx.x < WARP_SIZE) {
    float partialSum = threadIdx.x < NUM_WARPS ? partialSums_s[threadIdx.x] : 0.0f;
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      float before = output_ref.fetch_add(partialSum, cuda::memory_order_relaxed);
      printf("block %u (elements %3u..%3u) adds %4g; fetch_add returned %4g\n",
             blockIdx.x,
             segment,
             segment + 2 * BLOCK_DIM - 1,
             partialSum,
             before);
    }
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

static std::vector<float> makeInput() {
  std::vector<float> input(INPUT_LENGTH);
  for (unsigned int i = 0; i < INPUT_LENGTH; ++i) {
    input[i] = static_cast<float>((i * 7 + 3) % DIGITS);
  }
  return input;
}

static void printInput(const std::vector<float>& input) {
  std::printf("input (%u elements):\n", INPUT_LENGTH);
  for (unsigned int i = 0; i < INPUT_LENGTH; ++i) {
    std::printf(" %3.0f", input[i]);
    if ((i + 1) % VALUES_PER_ROW == 0) {
      std::printf("\n");
    }
  }
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  CHECK(cudaMemset(output_d, 0, sizeof(float)));
  launch(kernel, dim3(NUM_BLOCKS), dim3(BLOCK_DIM), input_d, output_d, INPUT_LENGTH);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  const std::vector<float> input = makeInput();
  std::printf("%u blocks of BLOCK_DIM = %u threads, each reduces a segment of %u elements\n",
              NUM_BLOCKS,
              BLOCK_DIM,
              2 * BLOCK_DIM);
  printInput(input);
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, INPUT_LENGTH * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));
  CHECK(cudaMemcpy(input_d, input.data(), INPUT_LENGTH * sizeof(float), cudaMemcpyHostToDevice));

  const float printedSum = runKernel(reduce_kernel_printing, input_d, output_d);
  const float bookSum = runKernel(reduce_kernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(input.data(), INPUT_LENGTH);
  std::printf(
      "reduce_kernel: %g, printing copy: %g, sequential: %g\n", bookSum, printedSum, expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 08_multi_block_atomic.cu -o 08_multi_block_atomic && ./08_multi_block_atomic

**What to look at in the output:** The four block sums add up to 1148, and each `fetch_add` returns the running total before that block (on a GPU the blocks may arrive in any order).

## Step 9 of 12: `09_coarsened_kernel.cu`

`CoarsenedSumReductionKernel` with `COARSE_FACTOR` = 2: each thread first adds 4 elements BLOCK_DIM apart without any barrier, so the same 256 elements need 2 blocks instead of 4. Printed as in step 8. (slide: *Code: the coarsened kernel*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 09_coarsened_kernel.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

constexpr unsigned int WARP_SIZE = 32;
constexpr unsigned int FULL_WARP_MASK = 0xffffffff;
constexpr unsigned int BLOCK_DIM = 32;
constexpr unsigned int NUM_WARPS = BLOCK_DIM / WARP_SIZE;
constexpr unsigned int COARSE_FACTOR = 2;
constexpr unsigned int NUM_BLOCKS = 2;
constexpr unsigned int INPUT_LENGTH = NUM_BLOCKS * COARSE_FACTOR * 2 * BLOCK_DIM;
constexpr unsigned int VALUES_PER_ROW = 16;
constexpr unsigned int DIGITS = 10;

__device__ unsigned int warpIdx() {
  return threadIdx.x / WARP_SIZE;
}

__device__ unsigned int laneIdx() {
  return threadIdx.x % WARP_SIZE;
}

__device__ __inline__ float warp_reduce(float val) {
  float partialSum = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialSum += __shfl_down_sync(FULL_WARP_MASK, partialSum, stride);
  }
  return partialSum;
}

__global__ void CoarsenedSumReductionKernel(float* input, float* output) {
  unsigned int segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = input[i];
  for (unsigned int c = 1; c < COARSE_FACTOR * 2; ++c) {
    partialSum += input[i + c * BLOCK_DIM];
  }

  partialSum = warp_reduce(partialSum);
  __shared__ float partialSums_s[NUM_WARPS];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    float partialSum = laneIdx() < NUM_WARPS ? partialSums_s[laneIdx()] : 0.0f;
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      output_ref.fetch_add(partialSum, cuda::memory_order_relaxed);
    }
  }
}

__global__ void CoarsenedSumReductionKernelPrinting(float* input, float* output) {
  unsigned int segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = input[i];
  for (unsigned int c = 1; c < COARSE_FACTOR * 2; ++c) {
    partialSum += input[i + c * BLOCK_DIM];
  }

  partialSum = warp_reduce(partialSum);
  __shared__ float partialSums_s[NUM_WARPS];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    float partialSum = laneIdx() < NUM_WARPS ? partialSums_s[laneIdx()] : 0.0f;
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      float before = output_ref.fetch_add(partialSum, cuda::memory_order_relaxed);
      printf("block %u (elements %3u..%3u) adds %4g; fetch_add returned %4g\n",
             blockIdx.x,
             segment,
             segment + COARSE_FACTOR * 2 * BLOCK_DIM - 1,
             partialSum,
             before);
    }
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

static std::vector<float> makeInput() {
  std::vector<float> input(INPUT_LENGTH);
  for (unsigned int i = 0; i < INPUT_LENGTH; ++i) {
    input[i] = static_cast<float>((i * 7 + 3) % DIGITS);
  }
  return input;
}

static void printInput(const std::vector<float>& input) {
  std::printf("input (%u elements):\n", INPUT_LENGTH);
  for (unsigned int i = 0; i < INPUT_LENGTH; ++i) {
    std::printf(" %3.0f", input[i]);
    if ((i + 1) % VALUES_PER_ROW == 0) {
      std::printf("\n");
    }
  }
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  CHECK(cudaMemset(output_d, 0, sizeof(float)));
  launch(kernel, dim3(NUM_BLOCKS), dim3(BLOCK_DIM), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  const std::vector<float> input = makeInput();
  std::printf("%u blocks of BLOCK_DIM = %u threads, COARSE_FACTOR = %u: each reduces %u elements\n",
              NUM_BLOCKS,
              BLOCK_DIM,
              COARSE_FACTOR,
              COARSE_FACTOR * 2 * BLOCK_DIM);
  printInput(input);
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, INPUT_LENGTH * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));
  CHECK(cudaMemcpy(input_d, input.data(), INPUT_LENGTH * sizeof(float), cudaMemcpyHostToDevice));

  const float printedSum = runKernel(CoarsenedSumReductionKernelPrinting, input_d, output_d);
  const float bookSum = runKernel(CoarsenedSumReductionKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(input.data(), INPUT_LENGTH);
  std::printf("CoarsenedSumReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 09_coarsened_kernel.cu -o 09_coarsened_kernel && ./09_coarsened_kernel

**What to look at in the output:** Two blocks of 128 elements each instead of four of 64, and the same total, 1148.

## Step 10 of 12: `10_reversed_kernel.cu`

Exercise 18.3, `ReversedSumReductionKernel`: thread t owns `input[8 + t]`, the stride halves 8, 4, 2, 1 and points left, and the active threads are the last `stride` ones, so they stay consecutive. Same input as step 3, printed after every stride. (slide: *Solutions 18.1, 18.2, 18.3, 18.6*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 10_reversed_kernel.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int N = 16;
constexpr unsigned int THREADS = N / 2;
const float INPUT[N] = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 8, 1, 9, 2, 4, 6};

__global__ void ReversedSumReductionKernel(float* input, float* output) {
  unsigned int i = blockDim.x + threadIdx.x;
  for (unsigned int stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x >= blockDim.x - stride) {
      input[i] += input[i - stride];
    }
    __syncthreads();
  }
  if (threadIdx.x == blockDim.x - 1) {
    *output = input[i];
  }
}

__device__ void printArray(const char* label,
                           unsigned int stride,
                           const float* values,
                           unsigned int n) {
  printf("%s %2u:", label, stride);
  for (unsigned int k = 0; k < n; ++k) {
    printf(" %3.0f", values[k]);
  }
  printf("\n");
}

__global__ void ReversedSumReductionKernelPrinting(float* input, float* output) {
  unsigned int i = blockDim.x + threadIdx.x;
  for (unsigned int stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x >= blockDim.x - stride) {
      input[i] += input[i - stride];
    }
    __syncthreads();

    if (threadIdx.x == 0) {
      printArray("after stride", stride, input, 2 * blockDim.x);
    }
    __syncthreads();
  }
  if (threadIdx.x == blockDim.x - 1) {
    *output = input[i];
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  CHECK(cudaMemcpy(input_d, INPUT, N * sizeof(float), cudaMemcpyHostToDevice));
  launch(kernel, dim3(1), dim3(THREADS), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  std::printf(
      "one block of %u threads, %u elements; thread t owns input[%u + t]\n", THREADS, N, THREADS);
  std::printf("%-15s", "input");
  for (unsigned int k = 0; k < N; ++k) {
    std::printf(" %3.0f", INPUT[k]);
  }
  std::printf("\n");
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, N * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));

  const float printedSum = runKernel(ReversedSumReductionKernelPrinting, input_d, output_d);
  const float bookSum = runKernel(ReversedSumReductionKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(INPUT, N);
  std::printf("ReversedSumReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);

  const bool ok = bookSum == expected && printedSum == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 10_reversed_kernel.cu -o 10_reversed_kernel && ./10_reversed_kernel

**What to look at in the output:** The mirror image of step 3: the partial sums stay packed on the right (positions 8 to 15 after stride 8, 12 to 15 after stride 4), the left half never changes, and `input[15]` ends at 62.

## Step 11 of 12: `11_coarsened_max_kernel.cu`

Exercise 18.4, `CoarsenedMaxReductionKernel`: the coarsened kernel with `fmaxf` instead of +, `warp_max`, the identity minus infinity (the host sets `*output = -INFINITY`, the input is all negative) and an atomic maximum for `float` written as a `compare_exchange_strong` loop. The printing copy shows each block's maximum and the value the loop found in `*output`. (slide: *Solutions 18.4 and 18.5*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 11_coarsened_max_kernel.cu
#include "cuda_compat.h"

#include <cmath>
#include <cstdio>
#include <vector>

constexpr unsigned int WARP_SIZE = 32;
constexpr unsigned int FULL_WARP_MASK = 0xffffffff;
constexpr unsigned int BLOCK_DIM = 32;
constexpr unsigned int NUM_WARPS = BLOCK_DIM / WARP_SIZE;
constexpr unsigned int COARSE_FACTOR = 2;
constexpr unsigned int NUM_BLOCKS = 2;
constexpr unsigned int INPUT_LENGTH = NUM_BLOCKS * COARSE_FACTOR * 2 * BLOCK_DIM;
constexpr unsigned int VALUES_PER_ROW = 16;
const float MAX_IDENTITY = -INFINITY;

__device__ unsigned int warpIdx() {
  return threadIdx.x / WARP_SIZE;
}

__device__ unsigned int laneIdx() {
  return threadIdx.x % WARP_SIZE;
}

__device__ float warp_max(float val) {
  float partialMax = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialMax = fmaxf(partialMax, __shfl_down_sync(FULL_WARP_MASK, partialMax, stride));
  }
  return partialMax;
}

__global__ void CoarsenedMaxReductionKernel(float* input, float* output) {
  unsigned int i = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x + threadIdx.x;
  float partialMax = input[i];
  for (unsigned int c = 1; c < COARSE_FACTOR * 2; ++c) {
    partialMax = fmaxf(partialMax, input[i + c * BLOCK_DIM]);
  }

  partialMax = warp_max(partialMax);
  __shared__ float partialMaxs_s[NUM_WARPS];
  if (laneIdx() == 0) {
    partialMaxs_s[warpIdx()] = partialMax;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    partialMax = warp_max(threadIdx.x < NUM_WARPS ? partialMaxs_s[threadIdx.x] : -INFINITY);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      float old = output_ref.load(cuda::memory_order_relaxed);
      while (old < partialMax &&
             !output_ref.compare_exchange_strong(old, partialMax, cuda::memory_order_relaxed)) {
      }
    }
  }
}

__global__ void CoarsenedMaxReductionKernelPrinting(float* input, float* output) {
  unsigned int segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialMax = input[i];
  for (unsigned int c = 1; c < COARSE_FACTOR * 2; ++c) {
    partialMax = fmaxf(partialMax, input[i + c * BLOCK_DIM]);
  }

  partialMax = warp_max(partialMax);
  __shared__ float partialMaxs_s[NUM_WARPS];
  if (laneIdx() == 0) {
    partialMaxs_s[warpIdx()] = partialMax;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    partialMax = warp_max(threadIdx.x < NUM_WARPS ? partialMaxs_s[threadIdx.x] : -INFINITY);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      float old = output_ref.load(cuda::memory_order_relaxed);
      while (old < partialMax &&
             !output_ref.compare_exchange_strong(old, partialMax, cuda::memory_order_relaxed)) {
      }
      printf("block %u (elements %3u..%3u) max %4g; output was %4g, %s\n",
             blockIdx.x,
             segment,
             segment + COARSE_FACTOR * 2 * BLOCK_DIM - 1,
             partialMax,
             old,
             old < partialMax ? "replaced by compare_exchange" : "kept");
    }
  }
}

float max_sequential(const float* input, unsigned int n) {
  float acc = MAX_IDENTITY;
  for (unsigned int i = 0; i < n; ++i) {
    acc = std::fmax(acc, input[i]);
  }
  return acc;
}

static std::vector<float> makeInput() {
  std::vector<float> input(INPUT_LENGTH);
  for (unsigned int i = 0; i < INPUT_LENGTH; ++i) {
    input[i] = static_cast<float>((i * 37 + 11) % 257) - 300.0f;
  }
  return input;
}

static void printInput(const std::vector<float>& input) {
  std::printf("input (%u elements, all negative):\n", INPUT_LENGTH);
  for (unsigned int i = 0; i < INPUT_LENGTH; ++i) {
    std::printf(" %4.0f", input[i]);
    if ((i + 1) % VALUES_PER_ROW == 0) {
      std::printf("\n");
    }
  }
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d) {
  CHECK(cudaMemcpy(output_d, &MAX_IDENTITY, sizeof(float), cudaMemcpyHostToDevice));
  launch(kernel, dim3(NUM_BLOCKS), dim3(BLOCK_DIM), input_d, output_d);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

int main() {
  const std::vector<float> input = makeInput();
  std::printf("%u blocks of BLOCK_DIM = %u threads, COARSE_FACTOR = %u: each reduces %u elements\n",
              NUM_BLOCKS,
              BLOCK_DIM,
              COARSE_FACTOR,
              COARSE_FACTOR * 2 * BLOCK_DIM);
  printInput(input);
  std::printf("the host sets *output = %g, the identity of max\n", MAX_IDENTITY);
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, INPUT_LENGTH * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));
  CHECK(cudaMemcpy(input_d, input.data(), INPUT_LENGTH * sizeof(float), cudaMemcpyHostToDevice));

  const float printedMax = runKernel(CoarsenedMaxReductionKernelPrinting, input_d, output_d);
  const float bookMax = runKernel(CoarsenedMaxReductionKernel, input_d, output_d);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = max_sequential(input.data(), INPUT_LENGTH);
  std::printf("CoarsenedMaxReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookMax,
              printedMax,
              expected);

  const bool ok = bookMax == expected && printedMax == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 11_coarsened_max_kernel.cu -o 11_coarsened_max_kernel && ./11_coarsened_max_kernel

**What to look at in the output:** Block 0's maximum, -45, replaces -inf, and block 1's, -44, replaces -45; on a GPU the blocks may arrive in the other order, and then the second one finds a larger value and keeps it. The result is -44 either way.

## Step 12 of 12: `12_any_length_kernel.cu`

Exercise 18.5, `CoarsenedSumReductionKernel` with a parameter `N`: every load is guarded by `i + c*BLOCK_DIM < N` and starts from the identity 0, the grid is `cdiv(N, COARSE_FACTOR*2*BLOCK_DIM)`, and no thread returns early, so the shuffles keep their full mask. It runs N = 256, 200, 129, 100 and 1 on buffers of exactly N elements. (slide: *Solutions 18.4 and 18.5*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 12_any_length_kernel.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

constexpr unsigned int WARP_SIZE = 32;
constexpr unsigned int FULL_WARP_MASK = 0xffffffff;
constexpr unsigned int BLOCK_DIM = 32;
constexpr unsigned int NUM_WARPS = BLOCK_DIM / WARP_SIZE;
constexpr unsigned int COARSE_FACTOR = 2;
constexpr unsigned int SEGMENT_LENGTH = COARSE_FACTOR * 2 * BLOCK_DIM;
constexpr unsigned int DIGITS = 10;
const unsigned int LENGTHS[] = {256, 200, 129, 100, 1};

__device__ unsigned int warpIdx() {
  return threadIdx.x / WARP_SIZE;
}

__device__ unsigned int laneIdx() {
  return threadIdx.x % WARP_SIZE;
}

__device__ __inline__ float warp_reduce(float val) {
  float partialSum = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialSum += __shfl_down_sync(FULL_WARP_MASK, partialSum, stride);
  }
  return partialSum;
}

__global__ void CoarsenedSumReductionKernel(float* input, float* output, unsigned int N) {
  unsigned int segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = 0.0f;
  for (unsigned int c = 0; c < COARSE_FACTOR * 2; ++c) {
    if (i + c * BLOCK_DIM < N) {
      partialSum += input[i + c * BLOCK_DIM];
    }
  }

  partialSum = warp_reduce(partialSum);
  __shared__ float partialSums_s[NUM_WARPS];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    float partialSum = laneIdx() < NUM_WARPS ? partialSums_s[laneIdx()] : 0.0f;
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      output_ref.fetch_add(partialSum, cuda::memory_order_relaxed);
    }
  }
}

__global__ void CoarsenedSumReductionKernelPrinting(float* input, float* output, unsigned int N) {
  unsigned int segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = 0.0f;
  for (unsigned int c = 0; c < COARSE_FACTOR * 2; ++c) {
    if (i + c * BLOCK_DIM < N) {
      partialSum += input[i + c * BLOCK_DIM];
    }
  }

  partialSum = warp_reduce(partialSum);
  __shared__ float partialSums_s[NUM_WARPS];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    float partialSum = laneIdx() < NUM_WARPS ? partialSums_s[laneIdx()] : 0.0f;
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      output_ref.fetch_add(partialSum, cuda::memory_order_relaxed);
      unsigned int segmentEnd = segment + SEGMENT_LENGTH < N ? segment + SEGMENT_LENGTH : N;
      printf("  block %u reads elements %3u..%3u (%3u of %u), adds %4g\n",
             blockIdx.x,
             segment,
             segmentEnd - 1,
             segmentEnd - segment,
             SEGMENT_LENGTH,
             partialSum);
    }
  }
}

float sum_sequential(const float* input, unsigned int n) {
  float sum = 0.0f;
  for (unsigned int i = 0; i < n; ++i) {
    sum += input[i];
  }
  return sum;
}

static std::vector<float> makeInput(unsigned int n) {
  std::vector<float> input(n);
  for (unsigned int i = 0; i < n; ++i) {
    input[i] = static_cast<float>((i * 7 + 3) % DIGITS);
  }
  return input;
}

template <class Kernel>
static float runKernel(Kernel kernel, float* input_d, float* output_d, unsigned int n) {
  CHECK(cudaMemset(output_d, 0, sizeof(float)));
  launch(kernel, dim3(cdiv(n, SEGMENT_LENGTH)), dim3(BLOCK_DIM), input_d, output_d, n);
  CHECK(cudaGetLastError());

  float output = 0.0f;
  CHECK(cudaMemcpy(&output, output_d, sizeof(float), cudaMemcpyDeviceToHost));
  return output;
}

static bool reduceLength(unsigned int n) {
  const std::vector<float> input = makeInput(n);
  std::printf("N = %u: cdiv(%u, %u) = %u block%s\n",
              n,
              n,
              SEGMENT_LENGTH,
              cdiv(n, SEGMENT_LENGTH),
              cdiv(n, SEGMENT_LENGTH) == 1 ? "" : "s");
  std::fflush(stdout);

  float* input_d = nullptr;
  float* output_d = nullptr;
  CHECK(cudaMalloc(&input_d, n * sizeof(float)));
  CHECK(cudaMalloc(&output_d, sizeof(float)));
  CHECK(cudaMemcpy(input_d, input.data(), n * sizeof(float), cudaMemcpyHostToDevice));

  const float printedSum = runKernel(CoarsenedSumReductionKernelPrinting, input_d, output_d, n);
  const float bookSum = runKernel(CoarsenedSumReductionKernel, input_d, output_d, n);
  CHECK(cudaFree(input_d));
  CHECK(cudaFree(output_d));

  const float expected = sum_sequential(input.data(), n);
  std::printf("  CoarsenedSumReductionKernel: %g, printing copy: %g, sequential: %g\n",
              bookSum,
              printedSum,
              expected);
  return bookSum == expected && printedSum == expected;
}

int main() {
  std::printf("BLOCK_DIM = %u threads, COARSE_FACTOR = %u: each block reduces up to %u elements\n",
              BLOCK_DIM,
              COARSE_FACTOR,
              SEGMENT_LENGTH);
  std::printf("input[i] = (7 i + 3) %% %u, and exactly N elements are allocated\n", DIGITS);

  bool ok = true;
  for (unsigned int n : LENGTHS) {
    ok = reduceLength(n) && ok;
  }

  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 12_any_length_kernel.cu -o 12_any_length_kernel && ./12_any_length_kernel

**What to look at in the output:** The last block reads only the elements that exist (72 of 128 for N = 200, 1 of 128 for N = 129), and every N matches the sequential sum.

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/08_reduction/README.md`.

**Next:** open `colab_2_full_programs.ipynb` in this same folder.